In [1]:
import sys
import os
import numpy as np
import torch

# Thêm đường dẫn để Python thấy cả 2 folder src
sys.path.append(os.path.abspath('..'))

# 1. Load dữ liệu giả lập
from src.utils import generate_simulated_data
data = generate_simulated_data()
X_cpu = data
X_gpu = torch.tensor(data, device='cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.float32)

# 2. Gọi hàm của BẠN
from src.model import hierarchical_kmeans_resampling
print("Đang chạy bản tái hiện của sinh viên...")
# Gọi hàm của BẠN với danh sách r_t_list [10, 5, 1]
centroids_mine = hierarchical_kmeans_resampling(
    X_cpu, 
    k_list=[3000, 1000, 300], 
    T=3, 
    m=10, 
    r_t_list=[2, 2, 2],
    num_init=1
)

# 3. Gọi hàm của TÁC GIẢ
# Lưu ý: Cần import đúng folder src_author
from src_author.hierarchical_kmeans_gpu import hierarchical_kmeans_with_resampling
print("Đang chạy bản gốc của tác giả...")
# Cấu hình tham số đúng như code tác giả yêu cầu
res_author = hierarchical_kmeans_with_resampling(
    data=X_gpu, 
    n_clusters=[3000, 1000, 300], 
    n_levels=3, 
    sample_sizes=[2, 2, 2], # r_t
    num_init=1,
    n_resamples=10,
    sample_strategy="closest"
)
centroids_author = res_author[-1]["centroids"].cpu().numpy()



Đang chạy bản tái hiện của sinh viên...
--- Level 1/3 (k=3000, r_t=2) ---
--- Level 2/3 (k=1000, r_t=2) ---


d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai

--- Level 3/3 (k=300, r_t=2) ---


d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai

Đang chạy bản gốc của tác giả...
Hierarchical k-means resampling steps: 100%|██████████| 10/10 [00:06<00:00,  1.65it/s]


In [2]:
import numpy as np
from sklearn.neighbors import KernelDensity

def calculate_kl_divergence(points, L=3, step=0.02, bandwidth=0.5):
    # Đảm bảo dữ liệu vào là float64
    points = points.astype(np.float64)
    
    x_range = np.arange(-L, L, step)
    y_range = np.arange(-L, L, step)
    gx, gy = np.meshgrid(x_range, y_range)
    grid = np.vstack([gx.ravel(), gy.ravel()]).T

    d_u = 1 / (4 * L**2)
    
    # KDE dùng float64
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian').fit(points)
    f = np.exp(kde.score_samples(grid))
    
    # Chuẩn hóa chuẩn theo diện tích delta (step^2)
    f_norm = f * 1 / (step**2 * f.sum())
    
    # Công thức KL chuẩn: sum( P * log(P/U) * delta ) hoặc tác giả dùng -sum( P * log(U/P) * delta )
    KL = -(f_norm * np.log(d_u / (f_norm + 1e-12))).sum() * (step**2)
    return KL

In [3]:
kl_mine = calculate_kl_divergence(centroids_mine)
kl_author = calculate_kl_divergence(centroids_author)

print(f"KL Sinh viên: {kl_mine:.4f}")
print(f"KL Tác giả: {kl_author:.4f}")

KL Sinh viên: 0.0492
KL Tác giả: 0.0405
